# <center> <img src="../notebooks/img/ITESOLogo.png" alt="ITESO" width="480" height="130"> </center>
# <center> **Departamento de Electrónica, Sistemas e Informática** </center>
---
## <center> **Big Data** </center>
---
### <center> **Spring 2026** </center>
---
### <center> **Pipeline de Streaming — NYC Taxi Trips** </center>
---
**Profesor**: Pablo Camarillo Ramirez  
**Alumno**: Emilio Daniel Guzmán Seda, Carlos Muñiz Lara

## Introducción

### Descripción de los Datos

Este pipeline procesa registros sintéticos de viajes de taxi de la ciudad de Nueva York (NYC Taxi Trips). Cada registro contiene **19 campos** que incluyen información temporal (pickup/dropoff), geográfica (LocationIDs), financiera (fare, tips, tolls) y operativa (VendorID, payment_type). El productor Kafka inyecta **datos sucios** de forma controlada:
- ~2% de registros con `trip_distance` negativa o cero
- ~1% de registros con `fare_amount` negativa o cero
- ~2% de registros con `PULocationID` o `DOLocationID` nulos
- `congestion_surcharge` y `airport_fee` siempre `None`
- ~1% de registros duplicados

### Diagrama de Arquitectura

```
┌─────────────────────┐         ┌─────────────────────┐         ┌─────────────────────┐
│                     │         │                     │         │                     │
│  kafka_producer.py  │────────▶│   Apache Kafka      │────────▶│  consumer.ipynb     │
│                     │  JSON   │   (kafka:9093)      │  Stream │  (PySpark SS)       │
│  - Genera viajes    │  UTF-8  │   Topic: taxi-trips │         │                     │
│  - Datos sucios     │         │                     │         │  foreachBatch:      │
│  - while True loop  │         └─────────────────────┘         │  ┌───────────────┐  │
│                     │                                         │  │ 1. Dedup      │  │
└─────────────────────┘                                         │  │ 2. Clean      │  │
                                                                │  │ 3. Join Zones │  │
                                ┌─────────────────────┐         │  │ 4. Aggregate  │  │
                                │                     │         │  │ 5. Write Neo4j│  │
                                │  taxi+_zone_lookup  │────────▶│  └───────────────┘  │
                                │  .csv (Static DF)   │  Join   │                     │
                                │  data/taxi_zones/   │         └──────────┬──────────┘
                                └─────────────────────┘                    │
                                                                           │ Neo4j Connector
                                                                           ▼
                                                                ┌─────────────────────┐
                                                                │                     │
                                                                │  Neo4j              │
                                                                │  (bolt://neo4j-     │
                                                                │   iteso:7687)       │
                                                                │                     │
                                                                │  (:Borough)         │
                                                                │  (:Trip)            │
                                                                │  [:PICKUP_IN]       │
                                                                │                     │
                                                                └─────────────────────┘
```

### Justificación Tecnológica

- **Apache Kafka**: Plataforma de mensajería distribuida que permite desacoplar el productor del consumidor. Ofrece durabilidad, escalabilidad horizontal y garantías de entrega at-least-once. Ideal para ingestión de datos en tiempo real.
- **Spark Structured Streaming**: Motor de procesamiento de streams que unifica batch y streaming bajo la misma API de DataFrames. Permite aplicar transformaciones complejas (joins, agregaciones, filtros) con semántica exactly-once mediante checkpoints.
- **Neo4j**: Base de datos de grafos nativa que permite modelar relaciones entre entidades (viajes y boroughs) de forma natural. El conector Spark-Neo4j facilita la escritura directa desde DataFrames usando Cypher.

In [13]:
# Instalar dependencias si es necesario
!pip install pyspark kafka-python numpy faker

26/05/11 01:52:00 WARN NetworkClient: [AdminClient clientId=adminclient-7] Connection to node 1001 (localhost/127.0.0.1:9092) could not be established. Node may not be available.
26/05/11 01:52:00 WARN NetworkClient: [AdminClient clientId=adminclient-8] Connection to node 1001 (localhost/127.0.0.1:9092) could not be established. Node may not be available.
26/05/11 01:52:01 WARN NetworkClient: [AdminClient clientId=adminclient-7] Connection to node 1001 (localhost/127.0.0.1:9092) could not be established. Node may not be available.
26/05/11 01:52:01 WARN NetworkClient: [AdminClient clientId=adminclient-8] Connection to node 1001 (localhost/127.0.0.1:9092) could not be established. Node may not be available.
26/05/11 01:52:02 WARN NetworkClient: [AdminClient clientId=adminclient-7] Connection to node 1001 (localhost/127.0.0.1:9092) could not be established. Node may not be available.
26/05/11 01:52:02 WARN NetworkClient: [AdminClient clientId=adminclient-8] Connection to node 1001 (local

26/05/11 01:52:00 WARN NetworkClient: [AdminClient clientId=adminclient-7] Connection to node 1001 (localhost/127.0.0.1:9092) could not be established. Node may not be available.
26/05/11 01:52:00 WARN NetworkClient: [AdminClient clientId=adminclient-8] Connection to node 1001 (localhost/127.0.0.1:9092) could not be established. Node may not be available.
26/05/11 01:52:01 WARN NetworkClient: [AdminClient clientId=adminclient-7] Connection to node 1001 (localhost/127.0.0.1:9092) could not be established. Node may not be available.
26/05/11 01:52:01 WARN NetworkClient: [AdminClient clientId=adminclient-8] Connection to node 1001 (localhost/127.0.0.1:9092) could not be established. Node may not be available.
26/05/11 01:52:02 WARN NetworkClient: [AdminClient clientId=adminclient-7] Connection to node 1001 (localhost/127.0.0.1:9092) could not be established. Node may not be available.
26/05/11 01:52:02 WARN NetworkClient: [AdminClient clientId=adminclient-8] Connection to node 1001 (local

26/05/11 01:52:00 WARN NetworkClient: [AdminClient clientId=adminclient-7] Connection to node 1001 (localhost/127.0.0.1:9092) could not be established. Node may not be available.
26/05/11 01:52:00 WARN NetworkClient: [AdminClient clientId=adminclient-8] Connection to node 1001 (localhost/127.0.0.1:9092) could not be established. Node may not be available.
26/05/11 01:52:01 WARN NetworkClient: [AdminClient clientId=adminclient-7] Connection to node 1001 (localhost/127.0.0.1:9092) could not be established. Node may not be available.
26/05/11 01:52:01 WARN NetworkClient: [AdminClient clientId=adminclient-8] Connection to node 1001 (localhost/127.0.0.1:9092) could not be established. Node may not be available.
26/05/11 01:52:02 WARN NetworkClient: [AdminClient clientId=adminclient-7] Connection to node 1001 (localhost/127.0.0.1:9092) could not be established. Node may not be available.
26/05/11 01:52:02 WARN NetworkClient: [AdminClient clientId=adminclient-8] Connection to node 1001 (local

26/05/11 01:52:08 WARN NetworkClient: [AdminClient clientId=adminclient-7] Connection to node 1001 (localhost/127.0.0.1:9092) could not be established. Node may not be available.


26/05/11 01:52:00 WARN NetworkClient: [AdminClient clientId=adminclient-7] Connection to node 1001 (localhost/127.0.0.1:9092) could not be established. Node may not be available.
26/05/11 01:52:00 WARN NetworkClient: [AdminClient clientId=adminclient-8] Connection to node 1001 (localhost/127.0.0.1:9092) could not be established. Node may not be available.
26/05/11 01:52:01 WARN NetworkClient: [AdminClient clientId=adminclient-7] Connection to node 1001 (localhost/127.0.0.1:9092) could not be established. Node may not be available.
26/05/11 01:52:01 WARN NetworkClient: [AdminClient clientId=adminclient-8] Connection to node 1001 (localhost/127.0.0.1:9092) could not be established. Node may not be available.
26/05/11 01:52:02 WARN NetworkClient: [AdminClient clientId=adminclient-7] Connection to node 1001 (localhost/127.0.0.1:9092) could not be established. Node may not be available.
26/05/11 01:52:02 WARN NetworkClient: [AdminClient clientId=adminclient-8] Connection to node 1001 (local

...

26/05/11 01:52:08 WARN NetworkClient: [AdminClient clientId=adminclient-8] Connection to node 1001 (localhost/127.0.0.1:9092) could not be established. Node may not be available.
26/05/11 01:52:09 WARN NetworkClient: [AdminClient clientId=adminclient-7] Connection to node 1001 (localhost/127.0.0.1:9092) could not be established. Node may not be available.
26/05/11 01:52:09 WARN NetworkClient: [AdminClient clientId=adminclient-8] Connection to node 1001 (localhost/127.0.0.1:9092) could not be established. Node may not be available.
26/05/11 01:52:10 WARN NetworkClient: [AdminClient clientId=adminclient-7] Connection to node 1001 (localhost/127.0.0.1:9092) could not be established. Node may not be available.
26/05/11 01:52:10 WARN NetworkClient: [AdminClient clientId=adminclient-8] Connection to node 1001 (localhost/127.0.0.1:9092) could not be established. Node may not be available.
26/05/11 01:52:10 WARN NetworkClient: [AdminClient clientId=adminclient-7] Connection to node 1001 (local

26/05/11 01:52:00 WARN NetworkClient: [AdminClient clientId=adminclient-7] Connection to node 1001 (localhost/127.0.0.1:9092) could not be established. Node may not be available.
26/05/11 01:52:00 WARN NetworkClient: [AdminClient clientId=adminclient-8] Connection to node 1001 (localhost/127.0.0.1:9092) could not be established. Node may not be available.
26/05/11 01:52:01 WARN NetworkClient: [AdminClient clientId=adminclient-7] Connection to node 1001 (localhost/127.0.0.1:9092) could not be established. Node may not be available.
26/05/11 01:52:01 WARN NetworkClient: [AdminClient clientId=adminclient-8] Connection to node 1001 (localhost/127.0.0.1:9092) could not be established. Node may not be available.
26/05/11 01:52:02 WARN NetworkClient: [AdminClient clientId=adminclient-7] Connection to node 1001 (localhost/127.0.0.1:9092) could not be established. Node may not be available.
26/05/11 01:52:02 WARN NetworkClient: [AdminClient clientId=adminclient-8] Connection to node 1001 (local

26/05/11 01:52:08 WARN NetworkClient: [AdminClient clientId=adminclient-7] Connection to node 1001 (localhost/127.0.0.1:9092) could not be established. Node may not be available.


26/05/11 01:52:08 WARN NetworkClient: [AdminClient clientId=adminclient-8] Connection to node 1001 (localhost/127.0.0.1:9092) could not be established. Node may not be available.
26/05/11 01:52:09 WARN NetworkClient: [AdminClient clientId=adminclient-7] Connection to node 1001 (localhost/127.0.0.1:9092) could not be established. Node may not be available.
26/05/11 01:52:09 WARN NetworkClient: [AdminClient clientId=adminclient-8] Connection to node 1001 (localhost/127.0.0.1:9092) could not be established. Node may not be available.
26/05/11 01:52:10 WARN NetworkClient: [AdminClient clientId=adminclient-7] Connection to node 1001 (localhost/127.0.0.1:9092) could not be established. Node may not be available.
26/05/11 01:52:10 WARN NetworkClient: [AdminClient clientId=adminclient-8] Connection to node 1001 (localhost/127.0.0.1:9092) could not be established. Node may not be available.
26/05/11 01:52:10 WARN NetworkClient: [AdminClient clientId=adminclient-7] Connection to node 1001 (local

26/05/11 01:52:12 WARN KafkaOffsetReaderAdmin: Error in attempt 2 getting Kafka offsets: 
java.util.concurrent.ExecutionException: org.apache.kafka.common.errors.TimeoutException: Timed out waiting for a node assignment. Call: listNodes
	at java.base/java.util.concurrent.CompletableFuture.reportGet(CompletableFuture.java:396)
	at java.base/java.util.concurrent.CompletableFuture.get(CompletableFuture.java:2073)
	at org.apache.kafka.common.internals.KafkaFutureImpl.get(KafkaFutureImpl.java:165)
	at org.apache.spark.sql.kafka010.ConsumerStrategy.retrieveAllPartitions(ConsumerStrategy.scala:65)
	at org.apache.spark.sql.kafka010.ConsumerStrategy.retrieveAllPartitions$(ConsumerStrategy.scala:64)
	at org.apache.spark.sql.kafka010.SubscribeStrategy.retrieveAllPartitions(ConsumerStrategy.scala:101)
	at org.apache.spark.sql.kafka010.SubscribeStrategy.assignedTopicPartitions(ConsumerStrategy.scala:112)
	at org.apache.spark.sql.kafka010.KafkaOffsetReaderAdmin.$anonfun$partitionsAssignedToAdmin$1(K

In [28]:
# Imports y creación de SparkSession
from spark_utils import SparkUtils
from pyspark.sql.functions import col, avg, sum, from_json
from pyspark.sql.types import StructType

# Paquetes necesarios: Kafka connector + Neo4j connector
packages = "org.apache.spark:spark-sql-kafka-0-10_2.13:4.0.0,org.neo4j:neo4j-connector-apache-spark_2.13:5.3.10_for_spark_3"

su = SparkUtils(
    spark_packages=packages
)



In [29]:
# Definición del esquema de los 19 campos del taxi NYC
columns_info = [
    ("VendorID", "long"),
    ("tpep_pickup_datetime", "timestamp"),
    ("tpep_dropoff_datetime", "timestamp"),
    ("passenger_count", "long"),
    ("trip_distance", "double"),
    ("RatecodeID", "long"),
    ("store_and_fwd_flag", "string"),
    ("PULocationID", "long"),
    ("DOLocationID", "long"),
    ("payment_type", "long"),
    ("fare_amount", "double"),
    ("extra", "double"),
    ("mta_tax", "double"),
    ("tip_amount", "double"),
    ("tolls_amount", "double"),
    ("improvement_surcharge", "double"),
    ("total_amount", "double"),
    ("congestion_surcharge", "double"),
    ("airport_fee", "double")
]

# Generar esquema StructType a partir de columns_info
taxi_schema = SparkUtils.generate_schema(columns_info)
taxi_schema

StructType([StructField('VendorID', LongType(), True), StructField('tpep_pickup_datetime', TimestampType(), True), StructField('tpep_dropoff_datetime', TimestampType(), True), StructField('passenger_count', LongType(), True), StructField('trip_distance', DoubleType(), True), StructField('RatecodeID', LongType(), True), StructField('store_and_fwd_flag', StringType(), True), StructField('PULocationID', LongType(), True), StructField('DOLocationID', LongType(), True), StructField('payment_type', LongType(), True), StructField('fare_amount', DoubleType(), True), StructField('extra', DoubleType(), True), StructField('mta_tax', DoubleType(), True), StructField('tip_amount', DoubleType(), True), StructField('tolls_amount', DoubleType(), True), StructField('improvement_surcharge', DoubleType(), True), StructField('total_amount', DoubleType(), True), StructField('congestion_surcharge', DoubleType(), True), StructField('airport_fee', DoubleType(), True)])

In [30]:
# Cargar DataFrame estático de zonas de taxi
zones_df = su._spark.read.csv(
    "/opt/spark/work-dir/data/taxi_zones/taxi+_zone_lookup.csv",
    header=True,
    inferSchema=True
)

zones_df.show(5)
zones_df.printSchema()

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
+----------+-------------+--------------------+------------+
only showing top 5 rows
root
 |-- LocationID: integer (nullable = true)
 |-- Borough: string (nullable = true)
 |-- Zone: string (nullable = true)
 |-- service_zone: string (nullable = true)



## Fase 1: Ingestión de Datos desde Kafka

In [31]:
# Lectura del stream desde Kafka y parseo de JSON
kafka_df = (su._spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "kafka:9093")
    .option("subscribe", "taxi-trips")
    .load())

# Convertir el valor binario de Kafka a string JSON
json_df = kafka_df.selectExpr("CAST(value AS STRING) as json_str")

# Parsear el JSON usando el esquema definido
df_parsed = (json_df
    .select(from_json(col("json_str"), taxi_schema).alias("data"))
    .select("data.*"))

df_parsed.printSchema()

root
 |-- VendorID: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)



## Fase 2: Función foreachBatch — Limpieza, Enriquecimiento y Persistencia

In [32]:
# Configuración de conexión a Neo4j
neo4j_url = "bolt://neo4j-iteso:7687"
neo4j_user = "neo4j"
neo4j_passwd = "neo4j@1234"

neo4j_options = {
    "url": neo4j_url,
    "authentication.basic.username": neo4j_user,
    "authentication.basic.password": neo4j_passwd,
    "batch.size": "5000",
    "transaction.retries": "3"
}


def process_micro_batch(batch_df, batch_id):
    """Procesa cada micro-batch: limpieza, enriquecimiento, agregación y escritura a Neo4j."""
    # 1. Verificar batch vacío
    if batch_df.isEmpty():
        print(f"Batch {batch_id}: vacío, saltando...")
        return

    print(f"--- Batch {batch_id}: {batch_df.count()} registros recibidos ---")

    # 2. Deduplicación
    df_clean = batch_df.dropDuplicates()

    # 3. Eliminar columnas innecesarias
    df_clean = df_clean.drop("congestion_surcharge", "airport_fee")

    # 4. Eliminar nulos en campos críticos
    df_clean = df_clean.dropna(subset=["PULocationID", "DOLocationID", "fare_amount"])

    # 5. Filtrar registros inválidos
    df_clean = df_clean.filter((col("trip_distance") > 0) & (col("fare_amount") > 0))

    if df_clean.isEmpty():
        print(f"Batch {batch_id}: sin registros válidos después de limpieza")
        return

    # 6. Stream-Static Join con zonas
    df_enriched = df_clean.join(zones_df, df_clean.PULocationID == zones_df.LocationID, "left") \
                          .withColumnRenamed("Borough", "Pickup_Borough") \
                          .drop("LocationID", "Zone", "service_zone")

    # 7. Filtrar registros sin borough (no match en join)
    df_with_borough = df_enriched.filter(col("Pickup_Borough").isNotNull())

    # 8. Escribir nodos Borough en Neo4j
    borough_df = df_with_borough.select(col("Pickup_Borough").alias("name")).distinct()
    borough_df.write \
        .format("org.neo4j.spark.DataSource") \
        .mode("Overwrite") \
        .options(**neo4j_options) \
        .option("labels", ":Borough") \
        .option("node.keys", "name") \
        .save()

    # 9. Escribir nodos Trip en Neo4j
    trip_df = df_with_borough.select(
        col("VendorID").alias("vendor_id"),
        col("tpep_pickup_datetime").cast("string").alias("pickup_dt"),
        col("tpep_dropoff_datetime").cast("string").alias("dropoff_dt"),
        "passenger_count", "trip_distance", "fare_amount",
        "tip_amount", "total_amount", "payment_type",
        col("Pickup_Borough").alias("pickup_borough")
    )
    trip_df.write \
        .format("org.neo4j.spark.DataSource") \
        .mode("Append") \
        .options(**neo4j_options) \
        .option("labels", ":Trip") \
        .option("node.keys", "vendor_id,pickup_dt,pickup_borough") \
        .save()

    # 10. Escribir relaciones PICKUP_IN con query Cypher
    rel_query = """
    MATCH (t:Trip {vendor_id: event.vendor_id, pickup_dt: event.pickup_dt, pickup_borough: event.pickup_borough})
    MATCH (b:Borough {name: event.pickup_borough})
    MERGE (t)-[:PICKUP_IN]->(b)
    """
    trip_df.write \
        .format("org.neo4j.spark.DataSource") \
        .mode("Append") \
        .options(**neo4j_options) \
        .option("query", rel_query) \
        .save()

    # 11. Calcular y mostrar agregaciones
    df_agg = df_with_borough.groupBy("Pickup_Borough", "VendorID", "payment_type") \
        .agg(
            avg("tip_amount").alias("avg_tip"),
            sum("total_amount").alias("total_revenue")
        )
    print(f"Batch {batch_id}: Agregaciones calculadas")
    df_agg.show(10, truncate=False)

## Fase 3: Ejecución del Stream

In [ ]:
from pathlib import Path
import shutil

# Limpieza previa del checkpoint (patrón de clase)
checkpoint_path = "/opt/spark/work-dir/checkpoints/taxi_streaming_checkpoint"
dir_path = Path(checkpoint_path)
if dir_path.exists() and dir_path.is_dir():
    shutil.rmtree(dir_path)

# Iniciar el stream con foreachBatch
query = (df_parsed.writeStream
    .foreachBatch(process_micro_batch)
    .option("checkpointLocation", checkpoint_path)
    .trigger(processingTime="10 seconds")
    .start())

print("Pipeline de streaming iniciado. Presiona Ctrl+C para detener.\n")
su._spark.streams.awaitAnyTermination()

26/05/11 02:29:25 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


Pipeline de streaming iniciado. Presiona Ctrl+C para detener.



StreamingQueryException: [STREAM_FAILED] Query [id = e7116f48-966a-42aa-9845-54de280a1c98, runId = ec93c89e-c4cc-471e-88c9-69550047b2fd] terminated with exception: [FOREACH_BATCH_USER_FUNCTION_ERROR] An error occurred in the user provided function in foreach batch sink. Reason: An exception was raised by the Python Proxy. Return Message: Traceback (most recent call last):
  File "/opt/spark/python/lib/py4j-0.10.9.9-src.zip/py4j/clientserver.py", line 644, in _call_proxy
    return_value = getattr(self.pool[obj_id], method)(*params)
  File "/opt/spark/python/pyspark/sql/utils.py", line 165, in call
    raise e
  File "/opt/spark/python/pyspark/sql/utils.py", line 162, in call
    self.func(DataFrame(jdf, wrapped_session_jdf), batch_id)
  File "/tmp/ipykernel_329/3029223225.py", line 56, in process_micro_batch
    .save()
  File "/opt/spark/python/pyspark/sql/readwriter.py", line 1743, in save
    self._jwrite.save()
  File "/opt/spark/python/lib/py4j-0.10.9.9-src.zip/py4j/java_gateway.p

--- Batch 1: 3 registros recibidos ---
--- Batch 12: 3 registros recibidos ---
Batch 0: vacío, saltando...


--- Batch 1: 2 registros recibidos ---


...

+--------------+--------+------------+-------+-------------+
|Pickup_Borough|VendorID|payment_type|avg_tip|total_revenue|
+--------------+--------+------------+-------+-------------+
|Manhattan     |1       |1           |7.12   |142.34       |
|Bronx         |1       |2           |8.49   |58.7         |
|Queens        |2       |3           |1.88   |77.38        |
|Manhattan     |1       |4           |3.67   |89.33        |
|Queens        |1       |2           |5.53   |149.69       |
|Bronx         |1       |1           |14.72  |170.77       |
|Staten Island |2       |1           |22.04  |61.92        |
|Queens        |2       |4           |6.08   |150.79       |
|Queens        |1       |3           |16.53  |92.33        |
+--------------+--------+------------+-------+-------------+

+--------------+--------+------------+-------+-------------+
|Pickup_Borough|VendorID|payment_type|avg_tip|total_revenue|
+--------------+--------+------------+-------+-------------+
|Manhattan     |1      

26/05/11 02:32:42 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 10000} milliseconds, but spent 14459 milliseconds
26/05/11 02:32:42 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 10000} milliseconds, but spent 14768 milliseconds


--- Batch 23: 8 registros recibidos ---
--- Batch 12: 8 registros recibidos ---


Batch 23: Agregaciones calculadas


Batch 12: Agregaciones calculadas


In [37]:
# Celda para verificación en Neo4j
print("--- Verificando datos en Neo4j ---")

# Contar nodos Borough
borough_count_df = su._spark.read.format("org.neo4j.spark.DataSource") \
    .options(**neo4j_options) \
    .option("query", "MATCH (b:Borough) RETURN count(b) AS count") \
    .load()
print("Conteo de :Borough")
borough_count_df.show()

# Contar nodos Trip
trip_count_df = su._spark.read.format("org.neo4j.spark.DataSource") \
    .options(**neo4j_options) \
    .option("query", "MATCH (t:Trip) RETURN count(t) AS count") \
    .load()
print("Conteo de :Trip")
trip_count_df.show()

# Contar relaciones PICKUP_IN
pickup_rel_count_df = su._spark.read.format("org.neo4j.spark.DataSource") \
    .options(**neo4j_options) \
    .option("query", "MATCH ()-[r:PICKUP_IN]->() RETURN count(r) AS count") \
    .load()
print("Conteo de [:PICKUP_IN]")
pickup_rel_count_df.show()

trip_df = su._spark.read.format("org.neo4j.spark.DataSource") \
    .options(**neo4j_options) \
    .option("query", """
        CALL {
            MATCH (t:Trip)
            RETURN t
            LIMIT 1
        }
        RETURN t
    """) \
    .load()

print("Datos de 1 trip:")
trip_df.show(truncate=False)

--- Verificando datos en Neo4j ---
Conteo de :Borough


+-----+
|count|
+-----+
|    8|
+-----+

Conteo de :Trip
+-----+
|count|
+-----+
| 3120|
+-----+

Conteo de [:PICKUP_IN]
+-----+
|count|
+-----+
| 3120|
+-----+

Datos de 1 trip:


+--------------------------------------------------------------------------------------------------------------+
|t                                                                                                             |
+--------------------------------------------------------------------------------------------------------------+
|{4, [Trip], 2, 6.0, 2026-04-24 21:48:06.966522, 4.2, 5, 26.03, 2026-04-24 21:51:06.966522, Brooklyn, 2, 21.03}|
+--------------------------------------------------------------------------------------------------------------+



In [ ]:
su.spark.stop()